# Problem 1: Steady One-Dimensional Convection-Diffusion Problem

In this problem, a scalar temperature field is transported by both convection and diffusion in a one-dimensional domain. Dirichlet boundary conditions are imposed at both ends of the domain.

The purpose of this problem is to compare the behavior of three convection schemes:

- Upwind Differencing Scheme (UDS)
- Central Differencing Scheme (CDS)
- QUICK scheme

The numerical solutions are compared with the analytical solution, and a grid-independence study is performed using 10, 20, 40, and 80 control volumes.

first of all we import python files provided in losson4 directory

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from Grid import Grid
from ScalarCoeffs import ScalarCoeffs
from BoundaryConditions import DirichletBc, BoundaryLocation
from LinearSolver import solve
from Models import DiffusionModel

## Advection Model

The provided framework contains models for diffusion, transient storage, and boundary conditions. However, it does not contain a model for convective transport.

To solve the convection-diffusion equation, an advection model is implemented below. The implementation follows the approach presented in Lesson 4, where the advection flux is evaluated at the cell faces and added to the finite-volume coefficient arrays.

Three convection schemes are considered:

- Upwind Differencing Scheme (UDS)
- Central Differencing Scheme (CDS)
- Quadratic Upwind Interpolation for Convective Kinematics (QUICK)

The UDS formulation is used for the implicit linearization because it guarantees bounded and stable coefficients. The CDS and QUICK schemes are implemented using deferred correction, allowing higher-order accuracy while preserving the stability of the UDS formulation.

In [ ]:
class AdvectionModel:
    """Class defining a 1D advection model.

    UDS is used for the linearization. CDS and QUICK are added using
    deferred correction so that the higher-order schemes can be compared
    while keeping the stable UDS coefficient structure.
    """

    def __init__(
        self,
        grid,
        phi,
        Uhe,
        rho,
        cp,
        west_bc,
        east_bc,
        scheme="UDS",
        west_alpha=1.0,
        east_alpha=-1.0,
    ):
        """Constructor.

        Parameters
        ----------
        grid : Grid
            One-dimensional finite-volume grid.
        phi : ndarray
            Scalar field including boundary values.
        Uhe : ndarray
            Velocity values at the faces.
        rho : float
            Density.
        cp : float
            Specific heat capacity.
        west_bc, east_bc : boundary condition objects
            Boundary conditions at west and east sides.
        scheme : str
            Advection scheme: 'UDS', 'CDS', or 'QUICK'.
        west_alpha, east_alpha : float
            Boundary alpha values used to force the correct boundary face value.
        """

        self._grid = grid
        self._phi = phi
        self._Uhe = Uhe
        self._rho = rho
        self._cp = cp
        self._west_bc = west_bc
        self._east_bc = east_bc
        self._scheme = scheme
        self._west_alpha = west_alpha
        self._east_alpha = east_alpha

        self._alphae = np.zeros(self._grid.ncv + 1)
        self._phie_uds = np.zeros(self._grid.ncv + 1)
        self._phie_hos = np.zeros(self._grid.ncv + 1)

    def add(self, coeffs):
        """Add advection terms to the coefficient arrays."""

        # Calculate UDS alpha values from the flow direction
        for i in range(self._grid.ncv + 1):
            if self._Uhe[i] >= 0:
                self._alphae[i] = 1.0
            else:
                self._alphae[i] = -1.0

        # Modify boundary alpha values as required by the assignment
        self._alphae[0] = self._west_alpha
        self._alphae[-1] = self._east_alpha

        # UDS face values
        self._phie_uds = (
            (1.0 + self._alphae) / 2.0 * self._phi[0:-1]
            + (1.0 - self._alphae) / 2.0 * self._phi[1:]
        )

        # Higher-order face values
        if self._scheme == "UDS":
            self._phie_hos[:] = self._phie_uds[:]

        elif self._scheme == "CDS":
            self._phie_hos[:] = 0.5 * (self._phi[0:-1] + self._phi[1:])

        elif self._scheme == "QUICK":
            self._phie_hos[:] = self._phie_uds[:]

            for i in range(1, self._grid.ncv):
                if self._Uhe[i] >= 0:
                    self._phie_hos[i] = (
                        -1.0 / 8.0 * self._phi[i - 1]
                        + 3.0 / 4.0 * self._phi[i]
                        + 3.0 / 8.0 * self._phi[i + 1]
                    )
                else:
                    self._phie_hos[i] = (
                        3.0 / 8.0 * self._phi[i]
                        + 3.0 / 4.0 * self._phi[i + 1]
                        - 1.0 / 8.0 * self._phi[i + 2]
                    )

        else:
            raise ValueError("Unknown scheme. Use 'UDS', 'CDS', or 'QUICK'.")

        # Use the UDS boundary values for all schemes
        self._phie_hos[0] = self._phie_uds[0]
        self._phie_hos[-1] = self._phie_uds[-1]

        # Face mass flow rates
        mdote = self._rho * self._Uhe * self._grid.Af

        # UDS fluxes for the implicit linearization
        flux_w_uds = self._cp * mdote[:-1] * self._phie_uds[:-1]
        flux_e_uds = self._cp * mdote[1:] * self._phie_uds[1:]

        # Higher-order fluxes for the residual
        flux_w_hos = self._cp * mdote[:-1] * self._phie_hos[:-1]
        flux_e_hos = self._cp * mdote[1:] * self._phie_hos[1:]

        # Mass imbalance correction
        imbalance = (
            -self._cp * mdote[1:] * self._phi[1:-1]
            + self._cp * mdote[:-1] * self._phi[1:-1]
        )

        # UDS linearization coefficients
        coeffW = -self._cp * mdote[:-1] * (1.0 + self._alphae[:-1]) / 2.0
        coeffE = self._cp * mdote[1:] * (1.0 - self._alphae[1:]) / 2.0
        coeffP = -coeffW - coeffE

        # Boundary coefficient treatment
        coeffP[0] += coeffW[0] * self._west_bc.coeff()
        coeffP[-1] += coeffE[-1] * self._east_bc.coeff()

        coeffW[0] = 0.0
        coeffE[-1] = 0.0

        # Deferred correction
        flux_uds = flux_e_uds - flux_w_uds
        flux_hos = flux_e_hos - flux_w_hos
        correction = flux_hos - flux_uds

        coeffs.accumulate_aP(coeffP)
        coeffs.accumulate_aW(coeffW)
        coeffs.accumulate_aE(coeffE)
        coeffs.accumulate_rP(flux_uds)
        coeffs.accumulate_rP(correction)
        coeffs.accumulate_rP(imbalance)

        return coeffs

## Problem Definition

We are solving the one-dimensional steady convection-diffusion equation across a domain of length
 $L = 1$.The boundary conditions are:$$T(0) = 0$$$$T(L) = 1$$We have chosen thermophysical properties so that the domain Peclet number is 50:$$Pe = \frac{\rho c_p u L}{k} = 50$$Because $Pe = 50$, this is a convection-dominated problem, making it an ideal test case to compare the performance of the UDS, CDS, and QUICK discretization schemes.

In [ ]:
# Domain parameters
L = 1.0
A = 1.0

# Grid
ncv = 10

# Flow properties
rho = 1.0
cp = 1.0
u = 1.0
k = 0.02

# Boundary temperatures
T1 = 0.0
T2 = 1.0

# Large timestep to obtain steady-state
dt = 1e10

# Solver controls
maxIter = 200
converged = 1e-8

## Analytical Solution

The analytical solution for the steady one-dimensional convection-diffusion problem is

$$
T(x) = T_1 + 
\frac{\exp(Pe x/L)-1}{\exp(Pe)-1}
(T_2-T_1)
$$

This analytical solution is used as the reference solution for the numerical results. The maximum error and RMS error are calculated by comparing the numerical solution at the cell centers with the analytical solution at the same locations.

In [ ]:
def analytical_solution(x, L, T1, T2, Pe):
    """Calculate the analytical solution for the 1D steady convection-diffusion problem."""
    
    return T1 + (T2 - T1) * (np.exp(Pe * x / L) - 1.0) / (np.exp(Pe) - 1.0)

## Numerical Solver Function

To make the results reproducible, the numerical solution is calculated inside a function. This avoids having results appear from code that was not clearly defined earlier in the notebook.

The function takes the number of control volumes and the advection scheme as inputs. It then builds the grid, applies the boundary conditions, adds the diffusion and advection models, and iterates until the residual is below the convergence tolerance.

In [ ]:
def solve_problem1(ncv, scheme, maxIter=500, converged=1e-10):
    """Solve Problem 1 for a given grid size and advection scheme."""

    # Create grid
    grid = Grid(
        lx=L,
        ly=ly,
        lz=lz,
        ncv=ncv
    )

    # Initialize coefficient object
    coeffs = ScalarCoeffs(grid.ncv)

    # Initialize scalar field including boundary values
    T = np.linspace(T1, T2, grid.ncv + 2)

    # Face velocity array
    Uhe = u * np.ones(grid.ncv + 1)

    # Define boundary conditions
    west_bc = DirichletBc(T, grid, T1, BoundaryLocation.WEST)
    east_bc = DirichletBc(T, grid, T2, BoundaryLocation.EAST)

    # Apply boundary conditions
    west_bc.apply()
    east_bc.apply()

    # Define models
    diffusion = DiffusionModel(grid, T, k, west_bc, east_bc)
    advection = AdvectionModel(
        grid=grid,
        phi=T,
        Uhe=Uhe,
        rho=rho,
        cp=cp,
        west_bc=west_bc,
        east_bc=east_bc,
        scheme=scheme,
        west_alpha=1.0,
        east_alpha=-1.0
    )

    # Nonlinear iteration
    for iteration in range(maxIter):
        coeffs.zero()

        coeffs = diffusion.add(coeffs)
        coeffs = advection.add(coeffs)

        maxResid = np.linalg.norm(coeffs.rP, np.inf)

        if maxResid < converged:
            break

        dT = solve(coeffs)

        T[1:-1] += dT

        west_bc.apply()
        east_bc.apply()

    # Analytical solution at cell centers
    T_exact = analytical_solution(grid.xP, L, T1, T2, Pe)

    # Error using internal cells only
    error = T[1:-1] - T_exact[1:-1]
    max_error = np.max(np.abs(error))
    rms_error = np.sqrt(np.mean(error**2))

    return grid, T, T_exact, max_error, rms_error, iteration + 1

## Comparison of Advection Schemes

The numerical solution is first calculated using 10 control volumes. The purpose of this test is to compare the behavior of the UDS, CDS, and QUICK schemes for a convection-dominated case with

$$
Pe = 50
$$

The same grid and physical properties are used for all three schemes.

In [ ]:
schemes = ["UDS", "CDS", "QUICK"]

results = {}

for scheme in schemes:
    grid, T, T_exact, max_error, rms_error, iterations = solve_problem1(
        ncv=10,
        scheme=scheme
    )
    
    results[scheme] = {
        "grid": grid,
        "T": T,
        "T_exact": T_exact,
        "max_error": max_error,
        "rms_error": rms_error,
        "iterations": iterations
    }

    print(f"{scheme}: iterations = {iterations}, max error = {max_error:.6e}, RMS error = {rms_error:.6e}")

## Plot of Numerical and Analytical Solutions

The plot below compares the numerical solutions from the three advection schemes with the analytical solution. Since the Peclet number is large, the solution remains almost constant over most of the domain and changes sharply near the outlet boundary.

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(grid.xP, T_exact, "k-", label="Analytical")

for scheme in schemes:
    plt.plot(
        results[scheme]["grid"].xP,
        results[scheme]["T"],
        marker="o",
        linestyle="--",
        label=scheme
    )

plt.xlabel("x")
plt.ylabel("T")
plt.title("Problem 1: Comparison of Advection Schemes, ncv = 10")
plt.legend()
plt.grid(True)
plt.show()

### Discussion of Results for 10 Control Volumes

For this case, the cell Peclet number is

$$
Pe_\Delta=\frac{50}{10}=5
$$

which indicates a strongly convection-dominated problem. The analytical solution remains close to zero throughout most of the domain and increases rapidly near the outlet.

All three schemes exhibit oscillations near the east boundary because the grid is too coarse to resolve the sharp outlet gradient. UDS converges rapidly but is more diffusive, while CDS and QUICK require more iterations due to the deferred-correction formulation.

Overall, 10 control volumes are insufficient to accurately resolve the solution, motivating the grid-independence study presented next.


## Grid-Independence Study

The calculation is repeated using

$$
N_{CV} = 10,\ 20,\ 40,\ 80
$$

The maximum error and RMS error are calculated for each grid. This provides a quantitative comparison between the numerical and analytical solutions, instead of relying only on plotted curves.

In [ ]:
ncv_values = [10, 20, 40, 80]

grid_study = {}

for scheme in schemes:
    grid_study[scheme] = {
        "ncv": [],
        "max_error": [],
        "rms_error": [],
        "iterations": []
    }

    for ncv in ncv_values:
        grid, T, T_exact, max_error, rms_error, iterations = solve_problem1(
            ncv=ncv,
            scheme=scheme
        )

        grid_study[scheme]["ncv"].append(ncv)
        grid_study[scheme]["max_error"].append(max_error)
        grid_study[scheme]["rms_error"].append(rms_error)
        grid_study[scheme]["iterations"].append(iterations)

       
        import pandas as pd

rows = []

for scheme in schemes:
    for i, ncv in enumerate(grid_study[scheme]["ncv"]):
        rows.append([
            scheme,
            ncv,
            grid_study[scheme]["max_error"][i],
            grid_study[scheme]["rms_error"][i],
            grid_study[scheme]["iterations"][i]
        ])

df = pd.DataFrame(
    rows,
    columns=[
        "Scheme",
        "NCV",
        "Max Error",
        "RMS Error",
        "Iterations"
    ]
)

df

## Error Comparison

The following plots show how the maximum error and RMS error change as the grid is refined. A decreasing error with grid refinement indicates that the numerical solution is approaching the analytical solution.

In [ ]:
plt.figure(figsize=(7, 5))

for scheme in schemes:
    plt.loglog(
        grid_study[scheme]["ncv"],
        grid_study[scheme]["max_error"],
        marker="o",
        label=scheme
    )

plt.xlabel("Number of control volumes")
plt.ylabel("Maximum absolute error")
plt.title("Grid Study: Maximum Error")
plt.legend()
plt.grid(True, which="both")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

for scheme in schemes:
    plt.loglog(
        grid_study[scheme]["ncv"],
        grid_study[scheme]["rms_error"],
        marker="o",
        label=scheme
    )

plt.xlabel("Number of control volumes")
plt.ylabel("RMS error")
plt.title("Grid Study: RMS Error")
plt.legend()
plt.grid(True, which="both")
plt.show()

### Discussion of Grid-Independence Results

The maximum and RMS errors decrease as the grid is refined, indicating convergence toward the analytical solution. The coarse-grid oscillations observed for 10 control volumes are significantly reduced on the finer meshes.

For the finest grid, QUICK produces the lowest RMS error, followed by CDS and UDS. This is consistent with the higher-order accuracy of QUICK and CDS compared with the first-order UDS scheme.

Overall, all schemes converge toward the analytical solution as the grid is refined, with QUICK providing the most accurate results on the finest mesh.


## Comparison of Advection Schemes for 80 Control Volumes

The numerical solutions obtained using 80 control volumes are shown below. 

In [ ]:
schemes = ["UDS", "CDS", "QUICK"]

results = {}

for scheme in schemes:
    grid, T, T_exact, max_error, rms_error, iterations = solve_problem1(
        ncv=80,
        scheme=scheme
    )
    
    results[scheme] = {
        "grid": grid,
        "T": T,
        "T_exact": T_exact,
        "max_error": max_error,
        "rms_error": rms_error,
        "iterations": iterations
    }
    plt.figure(figsize=(7, 5))

plt.plot(grid.xP, T_exact, "k-", label="Analytical")

for scheme in schemes:
    plt.plot(
        results[scheme]["grid"].xP,
        results[scheme]["T"],
        marker="o",
        linestyle="--",
        label=scheme
    )

plt.xlabel("x")
plt.ylabel("T")
plt.title("Problem 1: Comparison of Advection Schemes, ncv = 80")
plt.legend()
plt.grid(True)
plt.show()

Compared with the 10-control-volume case, the oscillations are significantly reduced and the numerical solutions are much closer to the analytical solution.